In [1]:
import websocket
import json
import time
import random
import string
import requests

def random_id(length=17):
    return ''.join(random.choices(string.ascii_letters + string.digits, k=length))

# Step 1: get sockjs session info
info = requests.get("https://www.sboulder.com/sockjs/info").json()
print("Server info:", info)

# Step 2: build session URL
server_id = str(random.randint(0, 999))
session_id = random_id(8)
ws_url = f"wss://www.sboulder.com/sockjs/{server_id}/{session_id}/websocket"

received = []

boulders = {}
counters = {}
def on_message_test(ws, message):
    received.append(message)
    print("RECV:", message[:200])

def on_message(ws, message):
    if message == "o":
        return  # SockJS open frame
    if message.startswith("a["):
        # SockJS array frame: a["<json_string>", "<json_string2>", ...]
        try:
            frames = json.loads(message[1:])  # strip leading 'a'
        except json.JSONDecodeError:
            return
        for frame in frames:
            try:
                data = json.loads(frame)
            except json.JSONDecodeError:
                continue

            msg_type = data.get("msg")

            if msg_type == "added" and data.get("collection") == "boulders":
                boulders[data["id"]] = data["fields"]

            elif msg_type == "changed" and data.get("collection") == "boulders":
                boulders.setdefault(data["id"], {}).update(data.get("fields", {}))

            elif msg_type == "added" and data.get("collection") == "counters-collection":
                counters[data["id"]] = data["fields"].get("count")

            elif msg_type == "ready":
                print("Subscription ready:", data.get("subs"))
                # If both subs are ready, we likely have everything
                if len(boulders) > 0:
                    print(f"\n--- Collected {len(boulders)} boulders ---")
                    print("Your sent count (countBoulders):", counters.get("countBoulders"))


def on_open(ws):
    print("Connected, sending DDP connect...")
    ws.send(json.dumps([json.dumps({
        "msg": "connect",
        "version": "1",
        "support": ["1", "pre2", "pre1"]
    })]))
    time.sleep(1)

    # Subscribe to boulders for a gym
    sub_id = random_id()
    ws.send(json.dumps([json.dumps({
        "msg": "sub",
        "id": sub_id,
        "name": "_boulders.list",
        "params": [
            {"gym": "arkose/montmartre", "isClosed": None},
            {"isClosed": 1, "createdAt": -1, "boulderNum": -1, "label": -1, "holdsColor": -1},
            20,
            None
        ]
    })]))

    # Subscribe to your personal sent count
    sub_id2 = random_id()
    ws.send(json.dumps([json.dumps({
        "msg": "sub",
        "id": sub_id2,
        "name": "_boulders.count",
        "params": [{"gym": "arkose/montmartre", "sentsList": "qQFsxQKYvqRqYJNKa", "isClosed": None}]
    })]))

def on_error(ws, error):
    print("ERROR:", error)

def on_close(ws, code, msg):
    print("Closed:", code, msg)

ws = websocket.WebSocketApp(
    ws_url,
    on_open=on_open,
    on_message=on_message,
    on_error=on_error,
    on_close=on_close,
    header={"Origin": "https://www.sboulder.com"}
)

ws.run_forever()

Server info: {'websocket': True, 'origins': ['*:*'], 'cookie_needed': False, 'entropy': 2804533838}
Connected, sending DDP connect...
Subscription ready: ['V236TOf6ec2Kdnn2J']

--- Collected 20 boulders ---
Your sent count (countBoulders): None
Subscription ready: ['WhkNljkvVGZCIxeGU']

--- Collected 20 boulders ---
Your sent count (countBoulders): 1
ERROR: 
Closed: None None


True

In [ ]:
YOUR_ID = "qQFsxQKYvqRqYJNKa"

your_sends = []
for bid, b in boulders.items():
    if YOUR_ID in b.get("sentsList", []):
        your_sends.append({
            "id": bid,
            "grade": b.get("grade"),
            "boulderNum": b.get("boulderNum"),
            "zone": b.get("zone"),
            "flashed": YOUR_ID in b.get("flashesList", []),
            "closedAt": b.get("closedAt"),
        })

for s in your_sends:
    print(s)